# 03. Query Rewriting 
— 검색 질문을 더 잘 만드는 Advanced RAG

## 이번 실습의 위치

앞 실습에서는 **문서를 더 잘 준비하는 방법**을 다뤘다.

```text
01 Naive RAG
   Plain Text + Recursive Chunking
            ↓
02 Document Processing
   Docling + Structure-aware Chunking + Metadata
            ↓
03 Query Rewriting
   사용자 질문 → 검색에 적합한 질문
```

이번 실습부터는 **Query-side Optimization**을 시작한다.

문서를 아무리 잘 나누고 임베딩해도 사용자의 질문이 모호하거나 구어체이면 검색 결과가 충분히 좋지 않을 수 있다.

예:

```text
사용자 질문
"AI가 회사 일에 어떻게 들어와요?"

검색용 질문
"2026년 AI 산업 전망에서 기업 운영에 AI 에이전트와
AI 기반 업무 자동화가 어떻게 도입될 것으로 전망되는가?"
```

## 학습 목표

1. Query Rewriting이 왜 필요한지 설명한다.
2. 원본 질문과 검색용 질문의 역할을 구분한다.
3. 원본 질문과 재작성 질문의 검색 결과를 비교한다.
4. Query Rewriting이 항상 검색 성능을 높이는 것은 아니라는 점을 이해한다.
5. 재작성 질문은 **검색에만 사용하고**, 최종 답변에는 원본 질문을 사용한다.

> 이번 실습에서는 Query Rewriting 하나에 집중한다.

# 1. 02번에서 만든 Vector DB 불러오기

02번에서 생성한 DB를 그대로 사용한다.

```text
chroma_db/ai_trend_report_doc_chunks
```

따라서 `02_document_processing_docling_chunks.ipynb`를 먼저 실행해야 한다.


In [1]:
import sys
# 상위 폴더의 common_config.py를 불러올 수 있도록 import 경로 추가
sys.path.append("..")
from common_config import embedding_model, llm_connect

from pathlib import Path
# LangChain용 Chroma 벡터 DB 래퍼 (02번에서 저장한 DB를 불러올 때 사용)
from langchain_chroma import Chroma

DB_PATH = Path("chroma_db/ai_trend_report_doc_chunks")
COLLECTION_NAME = "ai_trend_report_2026_doc_chunks"

# DB 폴더가 없으면 02번 노트북을 먼저 실행하라고 안내하고 중단
if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_docling_chunks.ipynb를 먼저 실행하여 "
        f"{DB_PATH}를 생성하세요."
    )

# 02번과 같은 임베딩 모델을 써야 저장된 벡터와 질문 벡터를 비교할 수 있음
embeddings = embedding_model()
# 질문 재작성과 답변 생성에 사용할 LLM (temperature=0: 결과를 최대한 일관되게)
llm = llm_connect(temperature=0, max_tokens=512)

# 02번에서 저장한 DB 폴더와 collection을 연결 (새로 임베딩하지 않고 재사용)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

# 불러온 chunk 수 확인 (02번에서 만든 개수와 같아야 정상)
print(f"불러온 chunk 수: {vector_store._collection.count()}")


불러온 chunk 수: 48


# 2. 먼저 문제를 확인해보기

Query Rewriting을 바로 적용하지 않는다.

먼저 **사용자 질문 그대로 검색했을 때 어떤 문제가 생기는지** 확인한다.

전문 문서에서는 다음 질문이 검색에 불리할 수 있다.

- 구어체 질문
- 너무 짧은 질문
- 문서에서 사용하는 용어와 다른 표현
- 질문 의도가 여러 방식으로 해석될 수 있는 표현

예:

```text
"AI가 회사 일에 어떻게 들어와요?"
```

사람은 의미를 이해하지만 검색 시스템에는 다음 정보가 부족하다.

```text
회사 일 → 기업 운영? 업무 자동화? 고객 지원?
AI → 생성형 AI? AI 에이전트? 산업 도입?
```


In [2]:
def inspect_retrieval(query: str, k: int = 3):
    """검색 결과를 metadata와 함께 확인한다.

    02번에서 Chroma collection을 cosine distance로 생성했으므로
    score는 낮을수록 질문과 문서가 더 가깝다.
    """
    # 질문을 임베딩해 가장 가까운 k개 chunk를 (문서, 거리) 쌍으로 가져오기
    # similarity_search_with_score: (Document, 거리) 튜플 리스트를 반환
    results = vector_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[검색 질문] {query}")
    print("cosine distance: 낮을수록 더 유사함")
    print("=" * 100)

    # 순위별로 거리, 페이지, chunk_id, 섹션(heading)과 본문 앞부분을 출력
    # 순위(1부터)별로 거리·메타데이터·본문 미리보기를 출력
    for rank, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata
        print(
            f"\n[{rank}] distance={score:.4f}"
            f" | pages={meta.get('pages')}"
            f" | chunk_id={meta.get('chunk_id')}"
        )
        print(f"heading={meta.get('heading')}")
        print("-" * 100)
        # 본문은 앞 600자만 보여주고, 줄바꿈은 공백으로 바꿔 한 줄에 표시
        print(doc.page_content[:600].replace("\n", " "))

    # 원본/재작성 결과를 비교할 수 있도록 반환
    return results


In [3]:
original_query = "AI가 회사 일에 어떻게 들어와요?"

# 재작성 전 원본 질문으로 검색 (비교 기준선)
original_results = inspect_retrieval(original_query, k=3)


[검색 질문] AI가 회사 일에 어떻게 들어와요?
cosine distance: 낮을수록 더 유사함

[1] distance=0.5251 | pages=4 | chunk_id=6
heading=AI@Data Report > 표·그림 목차
----------------------------------------------------------------------------------------------------
AI@Data Report 표·그림 목차 [표1], 1 = 분석 과정. [표1], 2 = 09. [표2], 1 = 분야별 데이터 수집 키워드. [표2], 2 = 10. [표3], 1 = 전처리 용어 정제 목록. [표3], 2 = 11. [표4], 1 = AI 산업기술정책 토픽모델링 종합 결과 및 시사점. [표4], 2 = 16. [그림1], 1 = 분야별 데이터셋 구성 예시. [그림1], 2 = 10. [그림2], 1 = 시장 규모 확대와 비즈니스 구조 전환 중심 워드 클라우드. [그림2], 2 = 13. [그림3], 1 = 기능 고도화와 적용 범위 확장 워드클라우드. [그림3], 2 = 14. [그림4], 1 = 안정성과 책임 중심의 규제 거버넌스 흐름 워드클라우드. [그림4], 2 = 15. [그림5], 1 = 주요 산업 분야별 AI 적용 사례. [그림5], 2 = 18. [그림6], 1 = AI규제 기준과 안정성 확보 기술의 대표 사례. [그림6], 2 = 19

[2] distance=0.5292 | pages=3 | chunk_id=5
heading=AI@Data Report > 기획 한국지능정보사회진흥원 인공지능데이터본부
----------------------------------------------------------------------------------------------------
AI@Data Report 기획 한국지능정보사회진흥원 인공지능데이터본부 신신애 본부장(sashin@nia.or.kr, 053-230-4200)

## 확인 포인트

검색 결과를 볼 때 처음부터 score만 보지 않는다.

1. **질문 의도에 맞는 문서가 검색되었는가?**
2. **원하는 section이 상위에 올라왔는가?**
3. page / heading metadata가 기대한 영역을 가리키는가?
4. 그 다음 distance score를 참고한다.

Advanced RAG에서 중요한 것은 단순히 숫자가 낮아졌는지가 아니라
**실제로 필요한 근거가 상위에 검색되는가**이다.


# 3. Query Rewriting

## Query Rewriting이란?

사용자의 질문 의도를 유지하면서 **검색 시스템이 찾기 쉬운 표현으로 바꾸는 기법**이다.

```text
사용자 질문
        ↓
      LLM
        ↓
검색에 적합한 질문
        ↓
    Retriever
```

중요한 점은 **새로운 질문을 만드는 것이 아니라 원래 의도를 보존하는 것**이다.

좋은 Query Rewriting:

```text
"AI가 회사 일에 어떻게 들어와요?"
        ↓
"2026년 AI 산업 전망에서 기업 운영에 AI 에이전트와
업무 자동화가 어떻게 도입될 것으로 전망되는가?"
```

나쁜 Query Rewriting:

```text
"AI가 회사 일에 어떻게 들어와요?"
        ↓
"2026년 제조업에서 로봇 AI가 생산성을 얼마나 높이는가?"
```

원본 질문에 없던 조건을 임의로 추가하면 질문의 의도가 변한다.


## 3.1 Rewriting Prompt 설계 원칙

초보자 실습에서는 아래 네 가지 규칙만 명확히 둔다.

1. 원본 질문의 의도를 유지한다.
2. 보고서 검색에 유용한 핵심 용어를 보완한다.
3. 원본 질문에 없는 사실이나 조건을 임의로 추가하지 않는다.
4. 검색용 질문 한 문장만 출력한다.


In [4]:
# LangChain 기본 구성요소: 프롬프트 템플릿 / LLM 출력을 문자열로 바꾸는 파서
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# system: 재작성 규칙 / human: 원본 질문({question})을 넣는 프롬프트
rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 RAG 검색용 Query Rewriting 도우미입니다.

사용자의 질문을 AI 트렌드 보고서에서 검색하기 좋은 질문으로 재작성하세요.

규칙:
1. 원본 질문의 핵심 의도를 반드시 유지하세요.
2. 검색에 도움이 되는 명확한 핵심 용어를 사용하세요.
3. 원본 질문에 없는 세부 사실이나 조건을 임의로 만들지 마세요.
4. 필요하면 'AI 산업', 'AI 기술', 'AI 정책', '2026년 전망',
   'AI 에이전트', '추론형 AI', '멀티모달', '규제 거버넌스'와 같은
   보고서 용어를 자연스럽게 사용하세요.
5. 최종 출력은 검색용 질문 한 문장만 작성하세요."""
    ),
    ("human", "원본 질문: {question}")
])

# 프롬프트 → LLM → 문자열 변환을 | 연산자로 연결한 체인
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


In [5]:
question = "AI가 회사 일에 어떻게 들어와요?"

# 원본 질문을 넣어 검색용 질문 한 문장 생성
rewritten_query = rewrite_chain.invoke({"question": question})

print("[원본 질문]")
print(question)

print("\n[재작성 질문]")
print(rewritten_query)


[원본 질문]
AI가 회사 일에 어떻게 들어와요?

[재작성 질문]
AI가 회사 업무에 어떻게 도입되고 활용되는지 설명해줘.


# 4. 원본 질문과 재작성 질문의 검색 결과 비교

Query Rewriting은 **재작성 문장 자체가 그럴듯한지 평가하는 것이 목적이 아니다.**

검색 결과가 실제로 좋아졌는지를 확인해야 한다.

```text
원본 질문 ─────→ Dense Retrieval ─→ Top-k
재작성 질문 ───→ Dense Retrieval ─→ Top-k
                         ↓
                     결과 비교
```


In [6]:
def compare_retrieval(original: str, rewritten: str, k: int = 3):
    print("\n" + "#" * 110)
    print("1. 원본 질문 검색")
    print("#" * 110)
    # 1) 원본 질문으로 Top-k 검색
    original_results = inspect_retrieval(original, k=k)

    print("\n" + "#" * 110)
    print("2. 재작성 질문 검색")
    print("#" * 110)
    # 2) 재작성 질문으로 Top-k 검색
    rewritten_results = inspect_retrieval(rewritten, k=k)

    # 두 결과를 반환해 이후 셀에서 Top-1 문서 변화 등을 직접 비교할 수 있게 함
    return original_results, rewritten_results


# 같은 k로 원본과 재작성 질문의 Top-k 결과를 나란히 비교
original_results, rewritten_results = compare_retrieval(
    original=question,
    rewritten=rewritten_query,
    k=3,
)



##############################################################################################################
1. 원본 질문 검색
##############################################################################################################
[검색 질문] AI가 회사 일에 어떻게 들어와요?
cosine distance: 낮을수록 더 유사함

[1] distance=0.5251 | pages=4 | chunk_id=6
heading=AI@Data Report > 표·그림 목차
----------------------------------------------------------------------------------------------------
AI@Data Report 표·그림 목차 [표1], 1 = 분석 과정. [표1], 2 = 09. [표2], 1 = 분야별 데이터 수집 키워드. [표2], 2 = 10. [표3], 1 = 전처리 용어 정제 목록. [표3], 2 = 11. [표4], 1 = AI 산업기술정책 토픽모델링 종합 결과 및 시사점. [표4], 2 = 16. [그림1], 1 = 분야별 데이터셋 구성 예시. [그림1], 2 = 10. [그림2], 1 = 시장 규모 확대와 비즈니스 구조 전환 중심 워드 클라우드. [그림2], 2 = 13. [그림3], 1 = 기능 고도화와 적용 범위 확장 워드클라우드. [그림3], 2 = 14. [그림4], 1 = 안정성과 책임 중심의 규제 거버넌스 흐름 워드클라우드. [그림4], 2 = 15. [그림5], 1 = 주요 산업 분야별 AI 적용 사례. [그림5], 2 = 18. [그림6], 1 = AI규제 기준과 안정성 확보 기술의 대표 사례. [그림6], 2 = 19

[2] distance=0.5292 | pages=3 | chunk_

## 4.1 비교할 때 학생들이 볼 것

- 원본 검색과 재작성 검색에서 **Top-1 문서가 달라졌는가?**
- 관련 section이 더 위로 올라왔는가?
- 관련 page가 Top-3 안에 들어왔는가?
- 재작성하면서 원래 질문의 의미가 변하지 않았는가?

> **score가 조금 좋아졌다고 무조건 좋은 Query Rewriting은 아니다.**  
> 관련 근거가 실제로 상위에 검색되었는지가 더 중요하다.


# 5. 여러 질문으로 Query Rewriting 연습

학생들은 먼저 **직접 검색용 질문을 작성한 뒤** LLM 결과와 비교한다.

```text
"AI가 회사 일에 어떻게 들어와요?"
"AI 규제는 왜 중요해졌나요?"
"AI가 생각하는 능력은 어떻게 좋아져요?"
"추론형 데이터는 어떤 종류가 있어요?"
```


In [7]:
practice_queries = [
    "AI가 회사 일에 어떻게 들어와요?",
    "AI 규제는 왜 중요해졌나요?",
    "AI가 생각하는 능력은 어떻게 좋아져요?",
    "추론형 데이터는 어떤 종류가 있어요?",
]

# 여러 질문을 같은 방식으로 재작성해 원본과 나란히 출력
for q in practice_queries:
    # 질문마다 LLM을 한 번씩 호출해 검색용 질문만 생성 (검색은 하지 않음)
    rewritten = rewrite_chain.invoke({"question": q})

    print("=" * 100)
    print(f"원본 질문  : {q}")
    print(f"재작성 질문: {rewritten}")
    print()


원본 질문  : AI가 회사 일에 어떻게 들어와요?
재작성 질문: AI가 회사 업무에 어떻게 도입되고 활용되나요?

원본 질문  : AI 규제는 왜 중요해졌나요?
재작성 질문: AI 규제와 규제 거버넌스가 AI 산업과 AI 기술 발전에서 왜 중요해졌는가?

원본 질문  : AI가 생각하는 능력은 어떻게 좋아져요?
재작성 질문: AI의 추론 능력은 어떤 기술 발전과 방법론으로 향상되나요?

원본 질문  : 추론형 데이터는 어떤 종류가 있어요?
재작성 질문: 추론형 데이터의 종류에는 어떤 것들이 있나요?



# 6. Query Rewriting의 실패 사례

Query Rewriting도 실패할 수 있다.

### 1. 의미 변경

```text
원본: AI 규제는 왜 중요해졌나요?
재작성: EU AI Act의 벌금 규모는 얼마인가요?
```

### 2. 과도한 확장

```text
원본: 추론형 데이터는 어떤 종류가 있어요?
재작성: 추론형 데이터의 종류, 구축 비용, 해외 시장 규모,
      주요 기업과 2030년 성장 전망은 무엇인가요?
```

### 3. 지나친 키워드 삽입

검색 키워드를 많이 넣는다고 항상 좋아지는 것은 아니다.

> **명확하게 만들되, 질문을 새로 만들지는 않는다.**


# 7. Query Rewriting을 RAG에 연결하기

## 검색에는 재작성 질문, 답변에는 원본 질문

```text
원본 질문
   │
   ├───────────────→ 최종 답변 질문으로 유지
   │
   ↓
Query Rewriting
   ↓
재작성 질문
   ↓
Retriever
   ↓
검색 문서
   │
   └───────────────┐
                   ↓
        원본 질문 + 검색 문서
                   ↓
                  LLM
                   ↓
               최종 답변
```

재작성 질문은 **검색을 위한 내부 표현**이다.


In [8]:
# 검색된 문서를 LLM 입력용 context 문자열로 변환 (출처 pages/heading 포함)
def format_docs(docs):
    parts = []

    for i, doc in enumerate(docs, start=1):
        # 검색 결과에 저장된 메타데이터(페이지, 섹션 heading)를 꺼냄
        meta = doc.metadata
        parts.append(
            f"[문서 {i} | pages={meta.get('pages')} | heading={meta.get('heading')}]\n"
            f"{doc.page_content}"
        )

    # 문서 사이를 빈 줄로 구분해 하나의 context 문자열로 합침
    return "\n\n".join(parts)


# 검색 문서 안에서만 답하도록 제한하는 답변 프롬프트 ({context}: 검색 결과, {question}: 원본 질문)
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """아래 검색 문서에 근거해서만 사용자의 질문에 답하세요.

규칙:
1. 검색 문서에서 확인되는 내용만 사용하세요.
2. 문서에서 확인할 수 없는 내용은 추측하지 마세요.
3. 핵심 내용을 이해하기 쉽게 설명하세요.

[검색 문서]
{context}"""
    ),
    ("human", "{question}")
])


In [9]:
def rewrite_rag(question: str, k: int = 4):
    # 1. 검색용 질문으로 재작성
    rewritten = rewrite_chain.invoke({"question": question})

    # 2. 재작성된 질문으로 검색
    docs = vector_store.similarity_search(rewritten, k=k)

    # 3. 검색 문서를 context로 구성
    context = format_docs(docs)

    # 4. 최종 답변은 원본 질문을 기준으로 생성
    # 답변 단계에는 원본 question을 넣는다 (재작성 질문은 검색 전용)
    messages = answer_prompt.invoke({
        "question": question,
        "context": context,
    })

    # 완성된 프롬프트 메시지를 LLM에 전달
    response = llm.invoke(messages)
    # 응답 객체에서 텍스트만 추출
    answer = StrOutputParser().invoke(response)

    return {
        "original_question": question,
        "rewritten_query": rewritten,
        "documents": docs,
        "answer": answer,
    }


In [10]:
# 원본 질문만 넣으면 재작성 → 검색 → 답변이 한 번에 실행됨
result = rewrite_rag(
    "AI가 회사 일에 어떻게 들어와요?",
    k=4,
)

print("[원본 질문]")
print(result["original_question"])

print("\n[검색용 재작성 질문]")
print(result["rewritten_query"])

# 최종 답변의 근거가 된 문서 출처 확인
print("\n[검색된 근거]")
# 최종 답변의 근거가 된 문서의 출처(pages, heading)를 확인
for i, doc in enumerate(result["documents"], start=1):
    print(
        f"{i}. pages={doc.metadata.get('pages')} "
        f"| heading={doc.metadata.get('heading')}"
    )

print("\n[최종 답변]")
print(result["answer"])


[원본 질문]
AI가 회사 일에 어떻게 들어와요?

[검색용 재작성 질문]
AI가 회사 업무에 어떻게 도입되고 활용되나요?

[검색된 근거]
1. pages=5 | heading=AI@Data Report > ■ 2025년 AI 산업·기술·정책 주요 담론
2. pages=3 | heading=AI@Data Report > 기획 한국지능정보사회진흥원 인공지능데이터본부
3. pages=4 | heading=AI@Data Report > 표·그림 목차
4. pages=5 | heading=AI@Data Report > ■ 분석 배경 및 목적

[최종 답변]
문서에 따르면, 회사 일에 AI가 들어오는 모습은 크게 다음처럼 볼 수 있습니다.

- **적용 범위가 넓어짐**: 기업과 산업 영역에서 AI의 적용 범위가 점점 넓어지고 있습니다.
- **도입이 더 깊어짐**: 단순히 한두 기능을 쓰는 수준을 넘어, 업무와 운영 체계 자체가 바뀌는 흐름이 나타납니다.
- **가치사슬 변화**: AI 도입으로 시장의 성장 속도뿐 아니라, **비즈니스 구조와 운영 방식**도 변하고 있습니다.
- **적용 환경이 다양해짐**: AI는 기존의 클라우드뿐 아니라 **온디바이스** 같은 환경으로도 확대되고 있습니다.

쉽게 말하면, AI는 회사에서 **업무 보조 도구**를 넘어서, **업무 방식과 운영 구조를 바꾸는 기술**로 들어오고 있다고 문서는 설명합니다.


# 8. 미니 실습 — 직접 판단해보기

다음 질문 중 하나를 선택한다.

```text
A. "AI 기술은 뭐가 좋아져요?"
B. "기업들이 AI를 왜 많이 써요?"
C. "AI 규제는 앞으로 어떻게 돼요?"
```

## 수행 순서

1. 원본 질문으로 Top-3 검색
2. 본인이 검색용 질문을 직접 작성
3. LLM으로 Query Rewriting 수행
4. 두 재작성 결과를 비교
5. 각각 Top-3 검색
6. page / heading / 내용 비교
7. 어느 질의가 가장 좋은 검색 결과를 만들었는지 설명

## 판단 기준

```text
[ ] 원본 질문의 의도를 유지했는가?
[ ] 문서의 핵심 용어와 가까워졌는가?
[ ] 관련 근거가 Top-k에 더 잘 들어왔는가?
[ ] 불필요한 조건을 새로 만들지 않았는가?
```


# 9. 선택 실습 — LCEL로 표현하기

처음 Advanced RAG를 학습하는 단계에서는
앞의 **단계별 Python 코드 흐름을 먼저 이해하는 것이 우선**이다.

LCEL은 구조를 이해한 뒤 선택적으로 확인한다.


In [11]:
# 선택 실습
from operator import itemgetter

from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Vector store를 retriever로 감싸고 검색 개수를 4로 설정
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 각 .assign()은 이전 결과를 이어받아 새 키를 하나씩 추가하는 단계
# .assign()을 이어 붙인 체인: 앞 단계 결과를 유지한 채 새 키를 하나씩 추가
# 흐름: question → rewritten_query → documents → context → answer
rewrite_rag_chain = (
    RunnablePassthrough.assign(
        # 1단계: 원본 question → 검색용 재작성 질문
        rewritten_query=itemgetter("question") | rewrite_chain
    )
    .assign(
        # 2단계: 재작성 질문 → retriever로 문서 검색
        documents=itemgetter("rewritten_query") | retriever
    )
    .assign(
        # 3단계: 검색 문서 → 프롬프트에 넣을 context 문자열
        context=itemgetter("documents") | RunnableLambda(format_docs)
    )
    .assign(
        # 4단계: 원본 question + context → 최종 답변 (재작성 질문은 답변에 넣지 않음)
        answer=(
            {
                "question": itemgetter("question"),
                "context": itemgetter("context"),
            }
            | answer_prompt
            | llm
            | StrOutputParser()
        )
    )
)

# 실행 예
# lcel_result = rewrite_rag_chain.invoke({
#     "question": "AI 규제는 왜 중요해졌나요?"
# })
# print(lcel_result["rewritten_query"])
# print(lcel_result["answer"])


# 10. 정리

## 오늘의 핵심

Query Rewriting은 **Pre-Retrieval 기법**이다.

```text
모호한 사용자 질문
       ↓
검색에 적합한 질문
       ↓
더 관련성 높은 후보 문서 검색
```

### 01 → 03까지의 발전

```text
01 Naive RAG
문서를 단순하게 나누고 검색
        ↓
02 Document Processing
문서 구조를 보존하여 검색 입력 품질 개선
        ↓
03 Query Rewriting
사용자 질문을 검색에 적합하게 개선
```

## 반드시 기억할 세 가지

1. **재작성 질문은 검색용이다.**
2. **최종 답변은 원본 질문에 답해야 한다.**
3. **Query Rewriting은 항상 좋아지는 것이 아니므로 검색 결과를 비교해야 한다.**

## 다음 실습

```text
04 Multi Query
1 Question → Multiple Queries
```